# Code to empirical test the 4th-Moment Jacobi method for generating Sigmapoints

In [1]:
import numpy as np
from compact_matrix_sqrt import *

In [6]:
n = 1000 # Dimension of matrices tested

In [9]:
U_dct = dct_matrix (n, make_orthogonal=True)
loss_dct = weighted_l4_loss (U_dct)
l4_bound_dct = ceol4_d_bound1 (U_dct)
print(f'DCT loss {loss_dct}')
print(f'conditional_expectation_signs bound for Q=DCT {l4_bound_dct}')

DCT loss 1.499
conditional_expectation_signs bound for Q=DCT 4.500000000000108


In [25]:
D_random = np.random.uniform (1,10, size=(n,))
U, sgns = conditional_expectation_signs (D_random*adversarial(U_dct), U_dct)
print (weighted_l4_loss(U))
#print (hist)

2.450218042839113


In [8]:
Sigma = random_SPSD (np.random.default_rng(), n)
loss, gain, U = optimize_l4 (Sigma, from_sigma=True, inplace=False, iterations=2)
err = np.linalg.norm (Sigma-U@U.T)
print(f'Total gain was {gain}, final value {loss}, theoretical bound 1, error {err}')
#print(U)

2.9990554488251666
1 0.11989648532784117 2.879158963497325
2 0.06398654320790544 2.81517242028942
4 0.04623020954758392 2.768942210741836
8 0.03607275323678338 2.7328694575050525
16 0.03186783261124488 2.7010016248938076
32 0.026012284906555265 2.6749893399872526
64 0.023166729893188465 2.6518226100940643
128 0.020532530499695893 2.6312900795943683
256 0.019856426553525625 2.6114336530408426
512 0.01757455644173783 2.593859096599105
1 0.014940963273843961 2.578918133325261
2 0.01352494612779969 2.565393187197461
4 0.012252348257861197 2.5531408389396
8 0.01091555645269956 2.5422252824869003
16 0.01011562897484567 2.532109653512055
32 0.008450857862688019 2.5236587956493666
64 0.007543319149268948 2.5161154765000977
128 0.0075859631442248355 2.5085295133558727
256 0.006233742517266892 2.502295770838606
512 0.005906121135682094 2.496389649702924
Total gain was 0.5026657991222426, final value 2.496389649702924, theoretical bound 1, error 2.127485448204479e-13


In [13]:
log = []
rng = np.random.default_rng()
dct = dct_matrix (n, True)
for ctr in range(50):
    Sigma = random_SPSD (rng, n)

    rand_Q = random_Q(rng, n)

    Uchol = np.linalg.cholesky (Sigma)
    loss_chol = weighted_l4_loss(Uchol)
    loss_random_chol = weighted_l4_loss (Uchol@rand_Q)
    loss_dct_chol = weighted_l4_loss (Uchol@dct)

    Usqrt = symmetric_matrix_sqrt (Sigma)
    loss_sqrt = weighted_l4_loss (Usqrt)
    loss_random_sqrt = weighted_l4_loss (Usqrt@rand_Q)
    loss_dct_sqrt = weighted_l4_loss (Usqrt@dct)

    adv_L = np.tril(rand_Q.T)
    loss_adv = weighted_l4_loss (adv_L@rand_Q)
    random_D = np.diag(rng.choice([-1.0, 1.0], size=n))
    loss_adv_random_d = weighted_l4_loss (adv_L@(random_D@rand_Q))
    rademacher_bound = rademacher_l4_bound (adv_L, rand_Q)

    U, _ = greedy_triangular_signs (adv_L, rand_Q)
    loss_adv_greedy = weighted_l4_loss (U)
    
    log.append([loss_chol, loss_random_chol, loss_dct_chol, 
                loss_sqrt, loss_random_sqrt, loss_dct_sqrt, 
                loss_adv, loss_adv_random_d, rademacher_bound, loss_adv_greedy])
print()
print (np.min (log, axis=0))
print (np.average (log, axis=0))
print (np.max (log, axis=0))
print ((np.max (log, axis=0)-np.min (log, axis=0))/np.max (log, axis=0))


[334.13007888   2.98336186   2.48726063 518.49086262   2.98224596
   2.21572852 334.02315884   2.99005201   3.10563504   2.97832847]
[335.62986283   2.99440945   2.49526266 519.62202883   2.9936699
   2.21960048 335.12379831   3.00721451   3.1129666    2.98936038]
[337.46493773   3.00339836   2.5010285  520.88884684   3.00242857
   2.22392071 336.26276262   3.05043993   3.12004153   3.0020413 ]
[0.00988209 0.00667128 0.00550488 0.00460364 0.0067221  0.00368367
 0.00666028 0.01979646 0.00461741 0.0078989 ]


In [8]:
Q  = dct_matrix (n, make_orthogonal=True)
adv_L = np.tril(Q.T)
print (weighted_l4_loss (adv_L@Q))
print (rademacher_l4_bound (adv_L, Q))

rho = q_square_row_coherence(Q)
bound = 3 * Q.shape[0] * rho
print (bound, rho*n)

results = [
    weighted_l4_loss (adv_L@np.diag(np.random.default_rng().choice([-1.0, 1.0], size=n))@Q) for ctr in range(100)
]
print(np.average(results))

387.24269141609443
3.0121036402317753
4.497000000000158 1.4990000000000527
2.9773648652242555


In [47]:
def test_conditional_expected_value():
    rng = np.random.default_rng(42)
    n = 8
    L = rng.normal (size=(n, n))
    Q = random_Q (rng, n)

    # Nichts fixiert
    value = conditional_expected_value(L, Q, -1, 0)

    row_norm2 = np.sum(L**2, axis=1)
    weights = 1.0 / row_norm2**2

    expected = np.sum(
        weights * 3.0 * row_norm2**2 / (n + 2)
    )

    np.testing.assert_allclose(value, expected, atol=1e-10)

    # Alles fixiert
    value = conditional_expected_value(L, Q, n - 1, n - 1)
    expected = weighted_l4_loss(L, Q)

    np.testing.assert_allclose(value, expected, atol=1e-10)

test_conditional_expected_value()

NameError: name 'conditional_expected_value' is not defined

## Conditional Q optimization (big algorithm)

### Generate a log for visualization

In [2]:
def log_Q ():
    n = 6
    Sigma = np.eye(6, dtype=np.float64)
    L, l4loss, history = optimize_conditional_Q_with_history (Sigma)
    print(L, l4loss)
    print(history)

log_Q()

[[ 0.54772256 -0.28786676 -0.21317683 -0.45066146 -0.60616864  0.03394406]
 [ 0.48304589  0.48750674  0.17808874 -0.46013659  0.49560926  0.199835  ]
 [ 0.41833001 -0.50391809  0.48867046  0.19034862  0.2777325  -0.4678673 ]
 [ 0.35355339  0.4201402  -0.52260775  0.37635316 -0.00589705 -0.53262263]
 [ 0.28867513 -0.37877071 -0.43367898  0.36408583  0.35456509  0.5717054 ]
 [ 0.28867513  0.3248868   0.47206451  0.52416198 -0.42900411  0.35983333]] 1.3200874756554544
[{'step': 0, 'step_typ': 'init', 'i': -1, 'j': -1, 'L': array([[1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 1.]]), 'Q': array([[1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 1.]]), 'econd': array([0.375, 0.375, 0.375, 0.375, 0.375, 0.375])}, {'step':